# Traffic Speed Prediction: Colab runner
Run the cells **top to bottom**. Before starting: **Runtime → Change runtime type → T4 GPU → Save**.

- Code comes from GitHub (`git clone` / `git pull`).
- Data and results live in **Google Drive** (`MyDrive/traffic_project/`) so they survive when Colab resets.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("PyTorch sees GPU:", torch.cuda.is_available())

## 2. Connect Google Drive (a pop-up asks for permission, allow it)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/traffic_project'
!mkdir -p {DRIVE}/data {DRIVE}/results

## 3. Get the latest code from GitHub

In [ ]:
REPO = 'https://github.com/pratyush069/traffic-speed-prediction.git'
%cd /content
!test -d traffic-speed-prediction || git clone {REPO}
%cd /content/traffic-speed-prediction
!git pull
# 'data' folder points to Google Drive, so the 414 MB download happens only once
!rm -rf data && ln -s {DRIVE}/data data
!git log --oneline -3

## 4. Install the extra libraries (PyTorch/XGBoost are already on Colab)

In [ ]:
!pip install -q -r requirements.txt

## 5. Prepare the data (first time: ~5 min; later sessions skip automatically)

In [ ]:
import os
if not os.path.exists('data/processed/speed.npz'):
    !python -m src.data.download
    !python -m src.data.preprocess
    !python -m src.data.graph
else:
    print('Data already prepared in Google Drive ✔')
!ls -lh data/processed

## 6. Quick test: XGBoost on the GPU (1 hour history → 30 min ahead)

In [ ]:
!python -m src.train --model xgboost --window 6 --horizon 3

## 7. Run experiments (add models here in later phases: `lstm`, `stgcn`)

In [ ]:
!python -m src.experiments --models persistence linear xgboost

## 8. Save results to Google Drive

In [ ]:
!cp results/all_results.csv {DRIVE}/results/all_results_colab.csv
import pandas as pd
pd.read_csv('results/all_results.csv').query("split == 'test'").round(3)